# Chapter 8: Time Series Analysis

Pandas has first-class support for time series data:

1. `pd.to_datetime` and `DatetimeIndex`
2. Resampling: `.resample()`
3. Rolling windows: `.rolling()`, `.expanding()`
4. Shifting: `.shift()`, `.diff()`, `.pct_change()`
5. Date components: `.dt` accessor
6. Practical example: stock price analysis

In [ ]:
import pandas as pd
import numpy as np

np.random.seed(42)

## 8.1 Creating Datetime Data

In [ ]:
# pd.to_datetime — parse strings to datetime
dates = pd.to_datetime(['2024-01-15', 'Jan 20, 2024', '2024/02/01', '03-15-2024'])
print('Parsed dates:', dates.tolist())

In [ ]:
# pd.date_range — generate date sequences
daily = pd.date_range('2024-01-01', periods=10, freq='D')
print('Daily:', daily.tolist()[:5])

monthly = pd.date_range('2024-01-01', periods=6, freq='MS')  # month start
print('Monthly:', monthly.tolist())

business = pd.date_range('2024-01-01', periods=10, freq='B')  # business days
print('Business:', business.tolist()[:5])

In [ ]:
# DatetimeIndex as DataFrame index
dates = pd.date_range('2023-01-01', periods=365, freq='D')
stock = pd.DataFrame({
    'price': 100 + np.cumsum(np.random.randn(365) * 2),
    'volume': np.random.randint(100000, 1000000, 365)
}, index=dates)

stock.head()

In [ ]:
# Datetime-aware indexing
print('March 2023:')
print(stock.loc['2023-03'].head())
print()
print('Q2 2023:')
print(stock.loc['2023-04':'2023-06'].head())

## 8.2 Resampling

Resampling changes the frequency of your time series.

- **Downsampling**: higher freq → lower (e.g., daily → monthly)
- **Upsampling**: lower freq → higher (e.g., monthly → daily)

In [ ]:
# Monthly average price
monthly_avg = stock['price'].resample('ME').mean()
print('Monthly average price:')
print(monthly_avg)

In [ ]:
# Weekly OHLC (Open, High, Low, Close)
weekly_ohlc = stock['price'].resample('W').ohlc()
weekly_ohlc.head()

In [ ]:
# Monthly aggregation with multiple metrics
monthly_stats = stock.resample('ME').agg({
    'price': ['mean', 'min', 'max'],
    'volume': 'sum'
})
monthly_stats.head()

## 8.3 Rolling Windows

Rolling windows compute statistics over a sliding window of fixed size.

In [ ]:
# 7-day and 30-day moving averages
stock['MA_7'] = stock['price'].rolling(window=7).mean()
stock['MA_30'] = stock['price'].rolling(window=30).mean()

# Also: rolling std for volatility
stock['volatility_30'] = stock['price'].rolling(window=30).std()

stock[['price', 'MA_7', 'MA_30', 'volatility_30']].tail(10)

In [ ]:
# Expanding window — cumulative statistics
stock['cum_mean'] = stock['price'].expanding().mean()
stock['cum_max'] = stock['price'].expanding().max()

stock[['price', 'cum_mean', 'cum_max']].tail(5)

## 8.4 Shifting: `.shift()`, `.diff()`, `.pct_change()`

In [ ]:
# .shift() — lag values
stock['prev_price'] = stock['price'].shift(1)  # yesterday's price

# .diff() — difference from previous value
stock['daily_change'] = stock['price'].diff()

# .pct_change() — percentage change
stock['daily_return'] = stock['price'].pct_change() * 100  # in percent

stock[['price', 'prev_price', 'daily_change', 'daily_return']].tail()

In [ ]:
# Returns statistics
returns = stock['daily_return'].dropna()
print(f'Average daily return: {returns.mean():.4f}%')
print(f'Return volatility:    {returns.std():.4f}%')
print(f'Best day:             {returns.max():.4f}%')
print(f'Worst day:            {returns.min():.4f}%')

## 8.5 Date Components: `.dt` Accessor

In [ ]:
# Create a Series with datetime values
dates = pd.Series(pd.date_range('2024-01-01', periods=100, freq='D'))

print('Year:       ', dates.dt.year.unique().tolist())
print('Month:      ', dates.dt.month.unique().tolist())
print('Day of week:', dates.dt.dayofweek.unique().tolist(), ' (0=Mon, 6=Sun)')
print('Day name:   ', dates.dt.day_name().unique().tolist())
print('Quarter:    ', dates.dt.quarter.unique().tolist())
print('Is weekend: ', dates.dt.dayofweek.isin([5, 6]).sum(), 'days')

## 8.6 Practical Example: Stock Price Analysis

In [ ]:
# Generate realistic-looking stock data
np.random.seed(123)
days = 500
dates = pd.date_range('2023-01-01', periods=days, freq='B')  # business days only

# Simulate random walk with drift
daily_returns = np.random.normal(0.0003, 0.015, days)  # slight upward drift
price = 150 * np.cumprod(1 + daily_returns)

stock_data = pd.DataFrame({
    'price': price,
    'volume': np.random.randint(500000, 5000000, days)
}, index=dates)

stock_data.head()

In [ ]:
# Add moving averages
stock_data['SMA_20'] = stock_data['price'].rolling(20).mean()
stock_data['SMA_50'] = stock_data['price'].rolling(50).mean()
stock_data['EMA_20'] = stock_data['price'].ewm(span=20).mean()

# Daily returns
stock_data['return'] = stock_data['price'].pct_change()

# 20-day rolling volatility (annualized)
stock_data['volatility'] = stock_data['return'].rolling(20).std() * np.sqrt(252)

stock_data.tail()

In [ ]:
# Monthly summary
monthly_summary = stock_data['price'].resample('ME').agg(
    ['first', 'last', 'min', 'max']
)
monthly_summary.columns = ['open', 'close', 'low', 'high']
monthly_summary['monthly_return'] = monthly_summary['close'].pct_change() * 100
monthly_summary.tail()

In [ ]:
# Best and worst months
print('Best month:')
print(monthly_summary.loc[monthly_summary['monthly_return'].idxmax()])
print()
print('Worst month:')
print(monthly_summary.loc[monthly_summary['monthly_return'].idxmin()])

In [ ]:
# Golden Cross / Death Cross signal
# Golden Cross: SMA_20 crosses above SMA_50 (bullish)
# Death Cross: SMA_20 crosses below SMA_50 (bearish)

stock_data['signal'] = np.where(
    stock_data['SMA_20'] > stock_data['SMA_50'], 'Bullish', 'Bearish'
)
stock_data['crossover'] = stock_data['signal'].ne(stock_data['signal'].shift())

crossovers = stock_data[stock_data['crossover'] & stock_data['SMA_50'].notna()]
print('Crossover events:')
crossovers[['price', 'SMA_20', 'SMA_50', 'signal']]

## 📝 Key Takeaways

| Concept | Key Point |
|---------|----------|
| `pd.to_datetime` | Parse strings to datetime |
| `DatetimeIndex` | Enables powerful time-based indexing |
| `resample()` | Change frequency (e.g., daily → monthly) |
| `rolling()` | Moving window statistics |
| `shift()` / `diff()` | Compare with previous values |
| `pct_change()` | Percentage returns |
| `.dt` accessor | Extract date components |